# 00 · Load and clean

Reads `Online Sales Data.xlsx`, applies the same cleaning rules as the Power BI model, and saves analysis-ready
parquet files to `analysis/data/`. Every later notebook starts from these files.

| Output | Grain | Notes |
|---|---|---|
| `customers.parquet` | 1 row per customer (301) | Deduplicated; adds lifetime orders, customer type, lifetime profit |
| `products.parquet` | 1 row per product (200) | As-is |
| `lines.parquet` | 1 row per order line (1,500) | Adds net sales, COGS, gross profit, revenue flag |
| `orders.parquet` | 1 row per order (405) | Order-level fields taken once (never summed) |

In [1]:
import sys, re
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.metrics import (load_raw, load_clean, dedupe_customers, reconcile, RAW_PATH,
                         DATA_DIR, OUTPUT_DIR, DATA_END, NON_REVENUE_STATUSES)
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
sns.set_theme(style="whitegrid", palette=["#0E7C86", "#F2A541", "#14213D", "#E4572E", "#5B6C8F"])
DATA_DIR.mkdir(parents=True, exist_ok=True); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUTPUT_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)

In [2]:
d = load_clean()
raw = d["raw"]
{k: v.shape for k, v in raw.items()}

{'Customers': (307, 12), 'Products': (200, 9), 'Order_Items': (1500, 15)}

## Reconcile with Power BI
Python totals must match the report before any analysis is trusted.

In [3]:
rec = reconcile(d["customers"], d["lines"])
assert rec["match"].all(), rec
rec

,python,power_bi,diff,match
net_sales,"108,222,683.47",108222683,0.47,True
gross_profit,"28,643,393.72",28643394,-0.28,True
revenue_orders,386.00,386,0.00,True
purchasing_customers,220.00,220,0.00,True
customers_clean,300.00,300,0.00,True


In [4]:
for name in ["customers", "products", "lines", "orders"]:
    d[name].to_parquet(DATA_DIR / f"{name}.parquet", index=False)
sorted(p.name for p in DATA_DIR.glob("*.parquet"))

['customers.parquet', 'lines.parquet', 'orders.parquet', 'products.parquet']

In [5]:
d['customers'].customer_type.value_counts()

customer_type
Recurring      116
One-Time       104
No Purchase     80
Name: count, dtype: int64